# Домашнее задание 3 — Агент-аналитик системных промптов (MCP + Skills)

### Задания

1. **Файловые инструменты агента** — реализовать инструменты для сохранения/чтения результатов анализа
2. **Пайплайн: агенты с tool loop** — реализовать агента и оркестратор пайплайна

### Критерии оценки (10 баллов)

| # | Критерий | Балл |
|---|----------|------|
| 1 | Реализованы все 5 файловых инструментов (`save_analysis`, `list_analyses`, `read_analysis`, `save_report`, `read_report`) + tool schemas + `LOCAL_TOOLS` | 1 |
| 2 | Написаны все 4 skill-файла (`skills.md`, `prompt_analysis.md`, `analysis_report.md`, `prompt_generator.md`) с двухуровневой структурой (Level 1 — метаданные в `skills.md`, Level 2 — полная процедура) | 1 |
| 3 | `prompt_analysis.md` содержит ≥5 критериев оценки промпта (Role/Persona, Few-shot, Chain-of-thought, Output format, Guardrails и т.д.) | 1 |
| 4 | Реализован `run_agent` с корректным tool loop: маршрутизация вызовов (MCP / `read_skill` / локальные), трассировка через `AgentTrace`, чистый контекст на каждый вызов | 1 |
| 5 | Сценарий `prompt_analysis` работает: агент загружает skill → читает файл из GitHub → анализирует → сохраняет через `save_analysis` | 1 |
| 6 | Сценарий `analysis_report` работает: агент читает все анализы → формирует сводный отчёт → сохраняет через `save_report` | 1 |
| 7 | Сценарий `prompt_generator` работает: агент читает отчёт → генерирует новый системный промпт → сохраняет результат | 1 |
| 8 | Progressive disclosure реализован как для skills, так и для MCP-инструментов | 1 |
| 9–10 | Сценарий `prompt_analysis` реализован через субагента (отдельный агентный вызов для каждого файла, а не один монолитный контекст) | 2 |

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh

downloading uv 0.11.3 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!


In [ ]:
!git clone https://github.com/GonnaMakeYouCry/HSE-Agent-Systems_2026.git

fatal: destination path 'HSE-Agent-Systems_2026' already exists and is not an empty directory.


In [ ]:
%cd HSE-Agent-Systems_2026

/content/HSE-Agent-Systems_2026


In [ ]:
!uv sync

Resolved 154 packages in 1ms
Checked 151 packages in 278ms


In [ ]:
!cp .env.example .env

---
## 0. Подготовка

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import sys
import json
import asyncio
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "pyproject.toml").exists():
            return p
    return start


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import nest_asyncio

nest_asyncio.apply()

from src.config import settings
from openai import OpenAI

In [ ]:
# ============================================================
# Настройка LLM-клиентов
# ============================================================

client = OpenAI(
    base_url="https://api.polza.ai/api/v1",
    api_key=settings.polza_ai_api_key,
)

MODEL = "openai/gpt-4o"

---
## 0.1 Утилиты для трассировки агента

Код из практики — необходим для выполнения заданий.

In [ ]:
# ============================================================
# Утилиты для трассировки агента (из практики 3)
# ============================================================
from dataclasses import dataclass, field


def _extract_usage(response) -> dict:
    """Извлечь токены из response.usage."""
    usage = getattr(response, "usage", None)
    if not usage:
        return {"prompt_tokens": 0, "cached_tokens": 0, "completion_tokens": 0, "total_tokens": 0, "context_size": 0}

    prompt_tokens = getattr(usage, "prompt_tokens", 0) or 0
    completion_tokens = getattr(usage, "completion_tokens", 0) or 0
    total_tokens = getattr(usage, "total_tokens", 0) or 0

    cached_tokens = getattr(usage, "prompt_cache_hit_tokens", 0) or 0
    if not cached_tokens:
        details = getattr(usage, "prompt_tokens_details", None)
        if details:
            cached_tokens = getattr(details, "cached_tokens", 0) or 0

    context_size = prompt_tokens + cached_tokens
    return {
        "prompt_tokens": prompt_tokens,
        "cached_tokens": cached_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": total_tokens,
        "context_size": context_size,
    }


@dataclass
class AgentStep:
    """Один шаг агента — для трассировки."""

    step_number: int
    tool_calls: list[dict] = field(default_factory=list)
    observations: list[dict] = field(default_factory=list)
    prompt_tokens: int = 0
    cached_tokens: int = 0
    context_size: int = 0
    context_size_delta: int = 0
    completion_tokens: int = 0
    total_tokens: int = 0
    final_answer: str | None = None


@dataclass
class AgentTrace:
    """Полный лог работы агента."""

    question: str
    steps: list[AgentStep] = field(default_factory=list)
    total_steps: int = 0
    prompt_tokens_total: int = 0
    completion_tokens_total: int = 0
    total_tokens_total: int = 0
    latest_context_size: int = 0
    max_context_size: int = 0
    final_answer: str = ""

    def summary(self) -> str:
        lines = [
            f"Вопрос: {self.question}",
            f"Шагов: {self.total_steps}",
            f"Prompt tokens (биллинг): {self.prompt_tokens_total}",
            f"Completion tokens: {self.completion_tokens_total}",
            f"Всего tokens: {self.total_tokens_total}",
            "",
        ]
        for s in self.steps:
            lines.append(f"--- Шаг {s.step_number} ---")
            if s.total_tokens:
                lines.append(
                    f"  \U0001f522 context={s.context_size} (\u0394 {s.context_size_delta:+}), "
                    f"completion={s.completion_tokens}"
                )
            for tc in s.tool_calls:
                lines.append(f"  \U0001f527 {tc['name']}({json.dumps(tc['args'], ensure_ascii=False)[:80]})")
            for obs in s.observations:
                text = str(obs)
                lines.append(f"  \U0001f4e1 {text[:120]}{'...' if len(text) > 120 else ''}")
            if s.final_answer:
                lines.append(f"  \U0001f4ac {s.final_answer[:300]}")
        lines.append(f"\n{'=' * 50}\n\U0001f3af Финальный ответ:\n{self.final_answer[:800]}")
        return "\n".join(lines)

---
## 0.2 Источник данных — GitHub MCP Server

Подключаемся к официальному GitHub MCP-серверу для доступа к репозиторию с системными промптами.

Репозиторий [`asgeirtj/system_prompts_leaks`](https://github.com/asgeirtj/system_prompts_leaks) — коллекция реальных системных промптов (ChatGPT, Claude, Gemini, Codex и др.)

### Предварительная настройка

#### 1. Node.js и npx

**npx** — утилита из экосистемы Node.js для запуска npm-пакетов без установки.

Проверить установку:
```bash
node --version   # должно быть v18+
npx --version    # должно быть 9+
```

#### 2. GitHub Personal Access Token

Создаётся в [Settings → Developer settings → Personal access tokens → Tokens (classic)](https://github.com/settings/tokens).

Минимальные права: `public_repo`.

Добавьте токен в файл `.env` в корне проекта:
```
GITHUB_TOKEN=ghp_ваш_токен_здесь
```

In [ ]:
!node --version   # должно быть v18+
!npx --version    # должно быть 9+

v20.19.0
10.8.2


In [ ]:
# ============================================================
# Подключение к GitHub MCP Server через stdio
# ============================================================
import os
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Параметры подключения — запускаем официальный GitHub MCP-сервер
github_server_params = StdioServerParameters(
    command="npx",
    args=["-y", "@modelcontextprotocol/server-github"],
    env={
        **os.environ,
        "GITHUB_PERSONAL_ACCESS_TOKEN": settings.github_token,
    },
)

In [ ]:
# ============================================================
# Discovery: подключаемся к GitHub MCP и узнаём его возможности
# ============================================================


async def discover_github_server():
    """Подключиться к GitHub MCP-серверу и получить список инструментов."""
    with open(os.devnull, "w", encoding="utf-8") as errlog:
        async with stdio_client(github_server_params, errlog=errlog) as (read_stream, write_stream):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()
                print("\u2705 Подключение к GitHub MCP установлено!\n")

                # Discovery: Tools
                tools_response = await session.list_tools()
                print(f"\U0001f527 Инструменты ({len(tools_response.tools)}):")
                print("-" * 55)
                for tool in tools_response.tools:
                    desc = (tool.description or "")[:80]
                    print(f"  \u2022 {tool.name}")
                    print(f"    {desc}...")
                    print()

                return tools_response.tools


github_tools = asyncio.get_event_loop().run_until_complete(discover_github_server())

✅ Подключение к GitHub MCP установлено!

🔧 Инструменты (26):
-------------------------------------------------------
  • create_or_update_file
    Create or update a single file in a GitHub repository...

  • search_repositories
    Search for GitHub repositories...

  • create_repository
    Create a new GitHub repository in your account...

  • get_file_contents
    Get the contents of a file or directory from a GitHub repository...

  • push_files
    Push multiple files to a GitHub repository in a single commit...

  • create_issue
    Create a new issue in a GitHub repository...

  • create_pull_request
    Create a new pull request in a GitHub repository...

  • fork_repository
    Fork a GitHub repository to your account or specified organization...

  • create_branch
    Create a new branch in a GitHub repository...

  • list_commits
    Get list of commits of a branch in a GitHub repository...

  • list_issues
    List issues in a GitHub repository with filtering options...

 

---
## 0.3 Skills — процедурные навыки агента

В практике мы создали skills для космического ассистента.  
Здесь используем **три навыка для анализа промптов**:

| Навык | Файл | Назначение |
|-------|------|------------|
| `prompt_analysis` | `hw_3_skills/prompt_analysis.md` | Пошаговый разбор системного промпта: секции, persona, приёмы, оценки |
| `analysis_report` | `hw_3_skills/analysis_report.md` | Формирование сводного отчёта с таблицами, цитатами и рекомендациями |
| `prompt_generator` | `hw_3_skills/prompt_generator.md` | Генерация нового промпта на основе выявленных паттернов |

### Skill Chaining

В практике skills были **независимые**. Здесь они образуют **цепочку** (chain):

```
prompt_analysis ──(структурированный разбор)──→ analysis_report
                                                       │
                                              (отчёт + рекомендации)
                                                       │
                                                       ▼
                                               prompt_generator
```

**Выход одного skill = вход следующего.** Агент сам решает, когда переключиться.

In [ ]:
from pathlib import Path
import textwrap

base_root = PROJECT_ROOT if "PROJECT_ROOT" in globals() else Path.cwd()
skills_dir = base_root / "lectures" / "lecture_3" / "hw_3_skills"
skills_dir.mkdir(parents=True, exist_ok=True)

files = {
    "skills.md": textwrap.dedent("""
    # Skills - Навыки агента для анализа системных промптов

    Набор skills, определяющих **как** агент должен действовать в типичных сценариях анализа системных промптов, подготовки сводного отчёта и генерации нового промпта на основе найденных паттернов.

    ## Список навыков

    | Навык | Файл | Краткое описание |
    |-------|------|------------------|
    | `prompt_analysis` | `prompt_analysis.md` | Пошаговый разбор одного системного промпта: структура, persona, инструкции, примеры, guardrails, формат ответа, сильные и слабые стороны |
    | `analysis_report` | `analysis_report.md` | Формирование сводного отчёта по нескольким анализам: общие паттерны, сравнительная таблица, рекомендации и выводы |
    | `prompt_generator` | `prompt_generator.md` | Генерация нового системного промпта на основе сводного отчёта и выявленных лучших практик |

    ## Как работают skills

    Каждый skill содержит:
    1. **Триггеры** - при каких запросах пользователя активируется навык
    2. **Протокол действий** - пошаговую процедуру, какие инструменты вызывать и в каком порядке
    3. **Формат вывода** - как оформлять результат
    4. **Переход к следующему skill** - когда результат текущего этапа становится входом следующего

    ## Политика progressive disclosure

    Агент **всегда** видит только этот файл (`skills.md`) как Level 1 - краткий обзор доступных навыков.

    Полную процедуру (Level 2) агент загружает **только по необходимости** через инструмент `read_skill(name)`.

    Это означает:
    - не загружай все skills заранее;
    - сначала определи, какой skill подходит под задачу;
    - затем вызови `read_skill(...)`;
    - после загрузки строго следуй процедуре выбранного skill.

    ## Когда какой skill использовать

    ### 1. `prompt_analysis`
    Активируй, когда пользователь хочет:
    - начать анализ одного или нескольких системных промптов;
    - разобрать конкретный prompt-файл из GitHub-репозитория;
    - получить структурированную оценку качества промпта;
    - понять, как устроены persona, guardrails, примеры, output format и workflow.

    **Выход skill:** сохранённый файл анализа в `hw_3_output/analyses/`.

    ---

    ### 2. `analysis_report`
    Активируй, когда пользователь хочет:
    - сформировать общий отчёт по уже сохранённым анализам;
    - выявить повторяющиеся паттерны;
    - сравнить подходы разных компаний / ассистентов;
    - получить сводные рекомендации.

    **Вход skill:** сохранённые анализы из `hw_3_output/analyses/`
    **Выход skill:** сохранённый отчёт в `hw_3_output/reports/analysis_report.md`

    ---

    ### 3. `prompt_generator`
    Активируй, когда пользователь хочет:
    - сгенерировать новый системный промпт;
    - получить шаблон промпта для нового ассистента;
    - собрать лучшие практики в единый prompt;
    - адаптировать найденные паттерны под новую роль или продукт.

    **Вход skill:** готовый отчёт `analysis_report`
    **Выход skill:** сохранённый результат в `hw_3_output/reports/generated_prompt.md`

    ## Skill Chaining

    skills образуют **цепочку**:

    ```text
    prompt_analysis ──(структурированный разбор)──→ analysis_report
                                                           │
                                                  (отчёт + рекомендации)
                                                           │
                                                           ▼
                                                   prompt_generator
    ```

    **Выход одного skill = вход следующего.** Агент сам решает, когда переключиться.

    ## Общие правила для всех skills

    - Не выдумывай содержимое prompt-файла - опирайся только на реально прочитанный текст.
    - Сначала получай данные инструментами, потом делай выводы.
    - Если задача многоэтапная - выполняй её последовательно.
    - Если анализируется несколько prompt-файлов, предпочитай **отдельный агентный вызов / субагента на каждый файл**, а не один переполненный контекст.
    - При цитировании используй короткие фрагменты и поясняй, что именно они показывают.
    - Сохраняй промежуточные результаты через файловые инструменты, а не держи всё только в контексте.
    """).strip() + "\n",

    "prompt_analysis.md": textwrap.dedent("""
    # Skill: Структурированный анализ системного промпта

    ## Требуемые инструменты
    - `read_skill`
    - GitHub MCP-инструмент чтения содержимого файла из репозитория
    - GitHub MCP-инструмент навигации / поиска по файлам репозитория
    - `save_analysis`

    ## Когда активировать
    Этот skill активируется, когда пользователь:
    - просит начать анализ системных промптов;
    - хочет разобрать один конкретный prompt-файл;
    - просит проанализировать prompt-репозиторий;
    - хочет понять, как устроен системный prompt у конкретного ассистента;
    - хочет получить оценку промпта по критериям качества.

    ## Протокол действий

    ### Шаг 1. Определи объект анализа
    Сначала пойми, что именно нужно анализировать:
    - один конкретный prompt-файл;
    - несколько prompt-файлов;
    - весь набор релевантных prompt-файлов в репозитории.

    Если пользователь не указал конкретный файл, сначала получи список релевантных файлов через GitHub / helper-инструмент навигации по репозиторию.

    ---

    ### Шаг 2. Используй progressive disclosure
    Не загружай лишние данные заранее.

    Правила:
    - сначала определи нужный skill;
    - затем работай только с тем prompt-файлом, который сейчас анализируешь;
    - не читай десятки файлов сразу;
    - не складывай в один контекст сырые тексты всех промптов.

    Если нужно проанализировать **несколько файлов**, выполняй анализ **по одному файлу за раз**, предпочтительно через **отдельный агентный вызов / субагента с чистым контекстом** на каждый файл.

    Это критично:
    - один файл -> один изолированный анализ;
    - один анализ -> одно сохранение через `save_analysis`;
    - только после этого переходи к следующему файлу.

    ---

    ### Шаг 3. Прочитай исходный prompt-файл
    Загрузи полный текст prompt-файла из GitHub-репозитория.

    После чтения:
    - зафиксируй путь файла;
    - определи компанию / продукт / ассистента;
    - определи основную задачу prompt-а;
    - проверь, есть ли в prompt-е явные секции.

    Не придумывай недостающие части. Если чего-то нет в тексте - так и напиши.

    ---

    ### Шаг 4. Выдели структуру промпта
    Сделай первичную декомпозицию prompt-а по секциям.

    Ищи и фиксируй:
    1. **Role / Persona** - кто этот ассистент, какая у него роль
    2. **Objective / Task** - что он должен делать
    3. **Capabilities / Scope** - что умеет и что не умеет
    4. **Workflow / Process** - есть ли пошаговая процедура
    5. **Output Format** - есть ли требования к структуре ответа
    6. **Guardrails / Safety** - ограничения, запреты, условия отказа
    7. **Examples / Few-shot** - есть ли демонстрационные примеры
    8. **Reasoning policy** - есть ли указания думать пошагово, проверять себя, не раскрывать chain-of-thought, сначала анализировать и потом отвечать
    9. **Tool-use policy** - есть ли правила использования инструментов / источников / файлов
    10. **Tone / Style** - тон, язык, степень формальности

    Если явных секций нет, восстанови их по смыслу.

    ---

    ### Шаг 5. Оцени prompt по критериям качества
    Обязательно оцени prompt как минимум по этим критериям:

    | Критерий | Что проверять |
    |----------|----------------|
    | **1. Role / Persona** | Насколько ясно задана роль ассистента |
    | **2. Task clarity** | Насколько конкретно описана задача |
    | **3. Workflow / decomposition** | Есть ли пошаговая процедура |
    | **4. Few-shot / examples** | Есть ли примеры желаемого поведения |
    | **5. Output format** | Есть ли явный формат результата |
    | **6. Guardrails / safety** | Есть ли ограничения, отказы, границы |
    | **7. Tool / source policy** | Понятно ли, когда и как использовать инструменты |
    | **8. Ambiguity handling** | Что делать при неясности, конфликтах, нехватке данных |
    | **9. Context discipline** | Есть ли указания не тащить лишний контекст, работать аккуратно |
    | **10. Robustness** | Насколько prompt устойчив к сложным / смешанным задачам |

    Для каждого критерия:
    - дай краткую оценку;
    - опиши сильную сторону;
    - опиши слабую сторону или риск;
    - при возможности приведи короткую цитату-подтверждение.

    Можно использовать шкалу 1–5 или словесную оценку (`сильно`, `средне`, `слабо`), но формат должен быть единообразным.

    ---

    ### Шаг 6. Извлеки паттерны проектирования prompt-а
    После критериальной оценки зафиксируй паттерны.

    Ищи:
    - сильную persona;
    - жёсткую маршрутизацию действий;
    - многошаговые инструкции;
    - правила работы с инструментами;
    - правила эскалации / отказа;
    - структурированные шаблоны ответа;
    - минимизацию галлюцинаций;
    - progressive disclosure;
    - clean context / изоляцию подзадач;
    - разделение между политикой поведения и источниками знаний.

    ---

    ### Шаг 7. Сформируй структурированный анализ
    Используй такой шаблон:

    ```text
    # Анализ промпта: [Название]

    ## 1. Метаданные
    - Источник: [путь в репозитории]
    - Компания / продукт:
    - Предполагаемая роль ассистента:
    - Основная задача prompt-а:

    ## 2. Краткое резюме
    2–5 предложений о том, как устроен этот prompt и чем он примечателен.

    ## 3. Структура prompt-а
    - Persona / Role:
    - Objectives:
    - Workflow:
    - Output format:
    - Guardrails:
    - Examples / Few-shot:
    - Tool policy:
    - Tone / Style:

    ## 4. Оценка по критериям
    | Критерий | Оценка | Наблюдение |
    |----------|--------|------------|
    | ... | ... | ... |

    ## 5. Сильные стороны
    - ...
    - ...
    - ...

    ## 6. Слабые стороны / риски
    - ...
    - ...
    - ...

    ## 7. Цитаты и подтверждения
    > короткая цитата
    Пояснение, что именно она демонстрирует.

    ## 8. Паттерны, которые стоит перенять
    - ...
    - ...

    ## 9. Что можно улучшить
    - ...
    - ...
    ```

    ---

    ### Шаг 8. Сохрани результат
    После завершения анализа обязательно вызови `save_analysis(name, content)`.

    Правила именования:
    - имя должно быть стабильным и удобным для чтения;
    - лучше использовать формат без пробелов и расширения;
    - если анализируется файл из папки в репозитории, можно включить в имя вендора / семейство модели.

    Примеры:
    - `OpenAI_Codex`
    - `Anthropic_Claude`
    - `Google_Gemini`

    ---

    ### Шаг 9. Если файлов несколько — повтори процесс изолированно
    Если пользователь попросил проанализировать несколько prompt-файлов:
    - не пиши один общий черновик по всем сразу;
    - для каждого файла запускай отдельный изолированный анализ;
    - сохраняй каждый анализ отдельно;
    - после завершения набора анализов переходи к skill `analysis_report`.

    ## Стиль ответа
    - Аналитический, структурированный, без воды
    - Сначала факты из prompt-а, потом интерпретация
    - Используй таблицы
    - Используй короткие цитаты как доказательства
    - Не выдумывай секции, которых нет
    - Явно отделяй наблюдения от рекомендаций

    ## Условие завершения
    Skill завершён, когда:
    1. prompt-файл прочитан;
    2. сделан структурированный разбор;
    3. выполнена оценка минимум по 5 критериям, лучше по 8–10;
    4. результат сохранён через `save_analysis`.
    """).strip() + "\n",

    "analysis_report.md": textwrap.dedent("""
    # Skill: Формирование сводного отчёта по анализам промптов

    ## Требуемые инструменты
    - `read_skill`
    - `list_analyses`
    - `read_analysis`
    - `save_report`

    ## Когда активировать
    Этот skill активируется, когда пользователь:
    - просит сформировать сводку по уже выполненным анализам;
    - хочет увидеть общие паттерны между prompt-ами;
    - просит сравнительный отчёт;
    - хочет получить рекомендации по лучшим практикам проектирования системных промптов;
    - завершил этап `prompt_analysis` и хочет перейти к синтезу.

    ## Протокол действий

    ### Шаг 1. Получи список доступных анализов
    Сначала вызови `list_analyses()` и проверь, есть ли сохранённые результаты.

    Если анализов нет:
    - честно сообщи, что сводный отчёт пока строить не из чего;
    - предложи сначала выполнить `prompt_analysis`.

    ---

    ### Шаг 2. Прочитай все сохранённые анализы
    Для каждого найденного анализа вызови `read_analysis(name)`.

    Правила:
    - прочитай **все** доступные анализы, если пользователь не ограничил выбор;
    - не опирайся только на память или название файла;
    - извлекай факты, таблицы, сильные стороны, слабые стороны и рекомендации из каждого анализа.

    ---

    ### Шаг 3. Нормализуй данные
    Приведи все анализы к единой рамке сравнения.

    Для каждого промпта зафиксируй:
    - объект анализа;
    - основную роль ассистента;
    - наличие / отсутствие persona;
    - наличие / отсутствие workflow;
    - наличие / отсутствие output format;
    - наличие / отсутствие guardrails;
    - наличие / отсутствие examples / few-shot;
    - наличие / отсутствие tool policy;
    - сильные стороны;
    - слабые стороны;
    - интересные паттерны.

    Если шкалы оценки различаются, переведи их к единому виду.

    ---

    ### Шаг 4. Построй сравнительную таблицу
    Сделай сводную таблицу по всем prompt-ам.

    Используй примерно такой формат:

    ```text
    | Prompt | Persona | Workflow | Output format | Guardrails | Examples | Tool policy | Общая сила |
    |--------|---------|----------|---------------|------------|----------|-------------|------------|
    | ...    | ...     | ...      | ...           | ...        | ...      | ...         | ...        |
    ```

    Таблица обязательна — она делает отчёт сравнимым и пригодным для следующего этапа.

    ---

    ### Шаг 5. Выдели повторяющиеся паттерны
    После таблицы сделай синтез.

    Обязательно выдели:
    1. Какие элементы встречаются почти у всех сильных prompt-ов
    2. Какие элементы чаще всего отсутствуют
    3. Какие конструкции выглядят особенно удачными
    4. Какие риски повторяются у нескольких prompt-ов
    5. Какие решения стоит перенять в новый prompt

    Ищи паттерны вроде:
    - сильная role / persona;
    - явное разбиение на шаги;
    - строгая политика инструментов;
    - требования к формату ответа;
    - инструкции по работе с неопределённостью;
    - guardrails и отказные правила;
    - приоритетность инструкций;
    - clean context / subagent strategy;
    - progressive disclosure.

    ---

    ### Шаг 6. Сформируй рекомендации
    На основе всех анализов сформулируй рекомендации двух уровней:

    #### A. Что обязательно включать в хороший системный prompt
    Например:
    - чёткую роль;
    - цель;
    - ограничения;
    - формат ответа;
    - правила для сложных задач;
    - политику использования инструментов.

    #### B. Чего лучше избегать
    Например:
    - расплывчатых инструкций;
    - отсутствия формата;
    - смешения политики и данных;
    - противоречивых требований;
    - слишком длинного монолитного контекста без маршрутизации.

    ---

    ### Шаг 7. Собери финальный отчёт
    Используй такой шаблон:

    ```text
    # Сводный отчёт по анализам системных промптов

    ## 1. Что было проанализировано
    Список файлов / ассистентов.

    ## 2. Краткий executive summary
    Короткий вывод на 5–10 предложений.

    ## 3. Сравнительная таблица
    | ... |

    ## 4. Общие сильные паттерны
    - ...
    - ...
    - ...

    ## 5. Повторяющиеся слабости и риски
    - ...
    - ...
    - ...

    ## 6. Лучшие практики для проектирования prompt-ов
    1. ...
    2. ...
    3. ...

    ## 7. Рекомендации для нового prompt-а
    - Какие секции нужны обязательно
    - Какие guardrails нужны обязательно
    - Как задавать output format
    - Как задавать tool policy
    - Как задавать поведение в неоднозначных случаях

    ## 8. Заключение
    Итоговый вывод.
    ```

    ---

    ### Шаг 8. Сохрани отчёт
    После формирования сводки обязательно вызови:

    - `save_report("analysis_report", content)`

    Если пользователь явно указал другое имя — используй его.

    ## Стиль ответа
    - Синтетический, сравнительный, структурированный
    - Меньше пересказа отдельных prompt-ов, больше общих выводов
    - Таблицы обязательны
    - Рекомендации должны быть конкретными и пригодными для следующего этапа
    - Пиши так, чтобы следующий skill (`prompt_generator`) мог использовать отчёт как вход

    ## Условие завершения
    Skill завершён, когда:
    1. получен список анализов;
    2. прочитаны все нужные анализы;
    3. построена сравнительная таблица;
    4. сформулированы паттерны и рекомендации;
    5. результат сохранён через `save_report`.
    """).strip() + "\n",

    "prompt_generator.md": textwrap.dedent("""
    # Skill: Генерация нового системного промпта на основе сводного отчёта

    ## Требуемые инструменты
    - `read_skill`
    - `read_report`
    - `save_report`

    ## Когда активировать
    Этот skill активируется, когда пользователь:
    - просит сгенерировать новый системный prompt;
    - хочет сделать prompt для нового ассистента на основе лучших практик;
    - хочет адаптировать выводы из `analysis_report` под новую задачу;
    - хочет получить итоговый production-style system prompt.

    ## Протокол действий

    ### Шаг 1. Прочитай сводный отчёт
    Сначала загрузи отчёт через `read_report("analysis_report")`, если пользователь не указал другое имя.

    Не генерируй новый prompt вслепую. Сначала извлеки:
    - лучшие практики;
    - обязательные секции;
    - паттерны guardrails;
    - удачные схемы output format;
    - правила tool use;
    - способы обработки неоднозначности.

    ---

    ### Шаг 2. Определи целевого ассистента
    Выясни, под какую задачу нужен новый prompt:
    - кто этот ассистент;
    - для каких пользователей;
    - какие задачи он решает;
    - есть ли инструменты;
    - какие ограничения безопасности нужны;
    - в каком тоне он должен отвечать.

    Если пользователь задал задачу не полностью, можно сделать разумный рабочий шаблон, но явно пометить допущения.

    ---

    ### Шаг 3. Собери каркас нового prompt-а
    Новый prompt должен быть не просто красивым текстом, а рабочей системой инструкций.

    Обязательные секции:
    1. **Role / Persona**
    2. **Mission / Objective**
    3. **Scope**
    4. **Workflow / Decision process**
    5. **Tool policy**
    6. **Output format**
    7. **Guardrails / Safety**
    8. **Ambiguity / clarification policy**
    9. **Error handling / fallback policy**
    10. **Tone / style**

    ---

    ### Шаг 4. Перенеси лучшие паттерны из отчёта
    При генерации consciously используй выводы из `analysis_report`.

    Примеры того, что нужно переносить:
    - чёткую роль;
    - многошаговые инструкции;
    - разделение между безопасным и небезопасным поведением;
    - явный формат ответа;
    - политику работы с инструментами;
    - правила эскалации / уточнения;
    - минимизацию галлюцинаций;
    - честность при нехватке данных;
    - clean context / изоляцию подзадач при сложных сценариях.

    Не копируй анализы механически. Синтезируй новый prompt.

    ---

    ### Шаг 5. Сформируй production-style prompt
    Используй такой шаблон:

    ```text
    # SYSTEM PROMPT

    ## Role
    [Кто ты]

    ## Mission
    [Главная задача]

    ## Core behavior
    - ...
    - ...
    - ...

    ## Workflow
    1. ...
    2. ...
    3. ...

    ## Tool usage policy
    - Когда использовать инструменты
    - Когда не использовать
    - Как проверять результаты
    - Как действовать при ошибке инструмента

    ## Output requirements
    - Формат ответа
    - Структура
    - Когда использовать списки / таблицы / код
    - Как обозначать неопределённость

    ## Safety and guardrails
    - Что запрещено
    - Когда отказывать
    - Как не выдумывать факты
    - Как не выходить за scope

    ## Ambiguity policy
    - Когда задавать уточняющий вопрос
    - Когда делать осторожное допущение
    - Как явно помечать допущения

    ## Final rule
    [Короткое приоритетное правило]
    ```

    ---

    ### Шаг 6. Добавь краткое пояснение
    Кроме самого prompt-а, дай короткое сопровождение:
    - какие паттерны были использованы;
    - почему структура получилась именно такой;
    - чем этот prompt должен быть лучше исходных.

    Но главный артефакт — именно **готовый system prompt**.

    ---

    ### Шаг 7. Сохрани результат
    После генерации обязательно вызови:

    - `save_report("generated_prompt", content)`

    Если пользователь указал другое имя — используй его.

    ---

    ### Шаг 8. Сделай результат удобным для повторного использования
    Финальный prompt должен:
    - быть самодостаточным;
    - не ссылаться на внутренние черновики;
    - быть пригодным для прямой вставки в system message;
    - иметь ясные заголовки;
    - не содержать лишних пояснений внутри самого prompt-а.

    ## Стиль ответа
    - Практичный, инженерный, без лишней воды
    - Сначала итоговый prompt
    - Затем короткое пояснение, какие решения были взяты из отчёта
    - Избегай абстрактных формулировок вроде “будь полезным” без конкретизации

    ## Условие завершения
    Skill завершён, когда:
    1. прочитан сводный отчёт;
    2. извлечены лучшие практики;
    3. сгенерирован полный новый system prompt;
    4. результат сохранён через `save_report`.
    """).strip() + "\n",
}

for filename, content in files.items():
    (skills_dir / filename).write_text(content, encoding="utf-8")

print(f"✅ Skills созданы в: {skills_dir}\n")
for path in sorted(skills_dir.glob("*.md")):
    print(f"• {path.name}")

print("\nПроверка содержимого папки:")
print([p.name for p in sorted(skills_dir.glob('*.md'))])

✅ Skills созданы в: /content/HSE-Agent-Systems_2026/lectures/lecture_3/hw_3_skills

• analysis_report.md
• prompt_analysis.md
• prompt_generator.md
• skills.md

Проверка содержимого папки:
['analysis_report.md', 'prompt_analysis.md', 'prompt_generator.md', 'skills.md']


In [ ]:
# ============================================================
# Реестр Skills — загружаем все skill-файлы из hw_3_skills/
# ============================================================

from pydantic import BaseModel, Field


class SkillInfo(BaseModel):
    """Метаданные одного skill-файла."""

    name: str = Field(description="имя файла (без .md)")
    title: str = Field(description="Заголовок skill")
    trigger_description: str = Field(description="Когда активировать (из секции 'Когда активировать')")
    content: str = Field(description="Полный текст skill")


def load_skills(skills_directory: Path) -> dict[str, SkillInfo]:
    """Загрузить все skill-файлы из директории (кроме skills.md)."""
    skills = {}

    for skill_file in sorted(skills_directory.glob("*.md")):
        if skill_file.name == "skills.md":
            continue

        with open(skill_file, "r", encoding="utf-8") as f:
            content = f.read()

        lines = content.split("\n")
        title = lines[0].replace("# Skill: ", "").strip() if lines else skill_file.stem

        trigger = ""
        in_trigger = False
        for line in lines:
            if "Когда активировать" in line:
                in_trigger = True
                continue
            if in_trigger:
                if line.startswith("##") and "Когда" not in line:
                    break
                trigger += line + "\n"

        name = skill_file.stem
        skills[name] = SkillInfo(
            name=name,
            title=title,
            trigger_description=trigger.strip(),
            content=content,
        )

    return skills


def load_skills_metadata(skills_directory: Path) -> str:
    """Загрузить skills.md — Level 1 метаданные для system prompt."""
    skills_md = skills_directory / "skills.md"
    if skills_md.exists():
        with open(skills_md, "r", encoding="utf-8") as f:
            return f.read()
    return "\u26a0\ufe0f Файл skills.md не найден в директории skills."


# --- Загрузка ---
skills_dir = PROJECT_ROOT / "lectures" / "lecture_3" / "hw_3_skills"
SKILLS = load_skills(skills_dir)

print(f"\U0001f4c2 Директория skills: {skills_dir}")
print(f"\U0001f4cb Загружено skills: {len(SKILLS)}\n")

for name, info in SKILLS.items():
    print(f"  \u2022 {name}: {info.title}")
    print(f"    Триггер: {info.trigger_description[:80]}...")
    print()

📂 Директория skills: /content/HSE-Agent-Systems_2026/lectures/lecture_3/hw_3_skills
📋 Загружено skills: 3

  • analysis_report: Формирование сводного отчёта по анализам промптов
    Триггер: Этот skill активируется, когда пользователь:
- просит сформировать сводку по уже...

  • prompt_analysis: Структурированный анализ системного промпта
    Триггер: Этот skill активируется, когда пользователь:
- просит начать анализ системных пр...

  • prompt_generator: Генерация нового системного промпта на основе сводного отчёта
    Триггер: Этот skill активируется, когда пользователь:
- просит сгенерировать новый систем...



In [ ]:
# ============================================================
# Level 1: загружаем skills.md — краткий обзор навыков для system prompt
# ============================================================

skills_metadata = load_skills_metadata(skills_dir)
print("Level 1 — содержимое skills.md (агент ВСЕГДА видит это в system prompt):\n")
print(skills_metadata)

Level 1 — содержимое skills.md (агент ВСЕГДА видит это в system prompt):

# Skills - Навыки агента для анализа системных промптов

Набор skills, определяющих **как** агент должен действовать в типичных сценариях анализа системных промптов, подготовки сводного отчёта и генерации нового промпта на основе найденных паттернов.

## Список навыков

| Навык | Файл | Краткое описание |
|-------|------|------------------|
| `prompt_analysis` | `prompt_analysis.md` | Пошаговый разбор одного системного промпта: структура, persona, инструкции, примеры, guardrails, формат ответа, сильные и слабые стороны |
| `analysis_report` | `analysis_report.md` | Формирование сводного отчёта по нескольким анализам: общие паттерны, сравнительная таблица, рекомендации и выводы |
| `prompt_generator` | `prompt_generator.md` | Генерация нового системного промпта на основе сводного отчёта и выявленных лучших практик |

## Как работают skills

Каждый skill содержит:
1. **Триггеры** - при каких запросах пользователя а

In [ ]:
# ============================================================
# Инструмент read_skill — загрузка процедуры по требованию
# ============================================================

READ_SKILL_TOOL_SCHEMA = {
    "type": "function",
    "function": {
        "name": "read_skill",
        "description": (
            "Загрузить детальную процедуру (skill) для решения задачи определённого типа. "
            "Вызывай этот инструмент, когда задача пользователя соответствует одному из "
            "доступных skills. После загрузки — строго следуй процедуре шаг за шагом."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "name": {
                    "type": "string",
                    "description": ("Имя skill для загрузки: " + ", ".join(SKILLS.keys())),
                }
            },
            "required": ["name"],
        },
    },
}


def read_skill(name: str) -> str:
    """Загрузить полный контент skill по имени."""
    if name in SKILLS:
        skill = SKILLS[name]
        return (
            f"\u2550\u2550\u2550 SKILL ЗАГРУЖЕН: {skill.title} \u2550\u2550\u2550\n\n"
            f"СТРОГО следуй этой процедуре шаг за шагом.\n\n"
            f"{skill.content}"
        )
    available = ", ".join(SKILLS.keys())
    return f"Skill '{name}' не найден. Доступные skills: {available}"


# --- Тест ---
print("\U0001f527 READ_SKILL_TOOL_SCHEMA:")
print(json.dumps(READ_SKILL_TOOL_SCHEMA, ensure_ascii=False, indent=2)[:500])
print("\n--- Тест: read_skill('prompt_analysis') ---")
print(read_skill("prompt_analysis")[:400] + "...")

🔧 READ_SKILL_TOOL_SCHEMA:
{
  "type": "function",
  "function": {
    "name": "read_skill",
    "description": "Загрузить детальную процедуру (skill) для решения задачи определённого типа. Вызывай этот инструмент, когда задача пользователя соответствует одному из доступных skills. После загрузки — строго следуй процедуре шаг за шагом.",
    "parameters": {
      "type": "object",
      "properties": {
        "name": {
          "type": "string",
          "description": "Имя skill для загрузки: analysis_report, prompt_a

--- Тест: read_skill('prompt_analysis') ---
═══ SKILL ЗАГРУЖЕН: Структурированный анализ системного промпта ═══

СТРОГО следуй этой процедуре шаг за шагом.

# Skill: Структурированный анализ системного промпта

## Требуемые инструменты
- `read_skill`
- GitHub MCP-инструмент чтения содержимого файла из репозитория
- GitHub MCP-инструмент навигации / поиска по файлам репозитория
- `save_analysis`

## Когда активировать
Этот skill активируется...


---
## 0.4 Вспомогательные функции пайплайна

Код из практики — утилиты для обхода репозитория и конвертации MCP-инструментов.

In [ ]:

def format_mcp_tools(mcp_tools) -> list[dict]:
    """Конвертировать MCP tools в OpenAI function-calling формат."""
    formatted = []
    for tool in mcp_tools:
        schema = tool.inputSchema if tool.inputSchema else {"type": "object", "properties": {}}
        formatted.append(
            {
                "type": "function",
                "function": {
                    "name": tool.name,
                    "description": tool.description or "",
                    "parameters": schema,
                },
            }
        )
    return formatted


---
## Задание 1: Файловые инструменты агента

Агент работает в несколько этапов, и между этапами ему нужно **сохранять и читать результаты** через файловую систему.

### Что нужно реализовать

| Инструмент | Назначение | Когда используется |
|------------|-----------|--------------------|
| `save_analysis(name, content)` | Сохранить анализ одного промпта в файл | После анализа каждого промпта |
| `list_analyses()` | Получить список всех сохранённых анализов | Перед формированием отчёта |
| `read_analysis(name)` | Прочитать конкретный анализ | При формировании отчёта |
| `save_report(name, content)` | Сохранить сводный отчёт / промпт | После формирования отчёта / промпта |
| `read_report(name)` | Прочитать сохранённый отчёт | Перед генерацией промпта |

### Файловая структура

```
hw_3_output/
  analyses/       ← save_analysis / read_analysis / list_analyses
    OpenAI_Codex.md
    Anthropic_Claude.md
    ...
  reports/         ← save_report / read_report
    analysis_report.md
    generated_prompt.md
```

### Подсказки
- Не забудьте добавить tool schemas в формате OpenAI function-calling
- Соберите все локальные инструменты в маппинг `LOCAL_TOOLS`

In [ ]:
# ============================================================
# Инструменты агента
# ============================================================

import json
import re

# --- Директории для результатов ---
ANALYSES_DIR = PROJECT_ROOT / "lectures" / "lecture_3" / "hw_3_output" / "analyses"
REPORTS_DIR = PROJECT_ROOT / "lectures" / "lecture_3" / "hw_3_output" / "reports"

ANALYSES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 Директория анализов: {ANALYSES_DIR}")
print(f"📂 Директория отчётов:  {REPORTS_DIR}")


# ============================================================
# Вспомогательные функции
# ============================================================

def _safe_name(name: str) -> str:
    """
    Нормализует имя файла:
    - убирает расширение .md, если пользователь его передал
    - заменяет пробелы на _
    - убирает опасные символы и path traversal
    """
    name = str(name).strip()
    name = name[:-3] if name.lower().endswith(".md") else name
    name = name.replace("\\", "/").split("/")[-1]   # защита от путей
    name = re.sub(r"\s+", "_", name)
    name = re.sub(r"[^\w\-.А-Яа-яЁё]", "_", name)
    name = re.sub(r"_+", "_", name).strip("._")
    return name or "untitled"


def _format_file_list(paths: list[Path], empty_message: str, title: str) -> str:
    """Красивый текстовый список markdown-файлов."""
    if not paths:
        return empty_message

    lines = [title]
    for path in sorted(paths, key=lambda p: p.name.lower()):
        size = path.stat().st_size
        lines.append(f"- {path.stem} ({size} байт)")
    return "\n".join(lines)


def _parse_mcp_json_text(raw_text: str):
    """
    MCP обычно возвращает текст, внутри которого лежит JSON.
    Пытаемся аккуратно его распарсить.
    """
    raw_text = (raw_text or "").strip()
    if not raw_text:
        return None

    # Если ответ завернут в ```json ... ```
    fenced = re.match(r"^```(?:json)?\s*(.*?)\s*```$", raw_text, flags=re.DOTALL)
    if fenced:
        raw_text = fenced.group(1).strip()

    try:
        return json.loads(raw_text)
    except Exception:
        return None


def _extract_entries_from_github_response(data) -> list[dict]:
    """
    Нормализует разные возможные форматы ответа get_file_contents(...) к списку entries.
    """
    if data is None:
        return []

    if isinstance(data, list):
        return [x for x in data if isinstance(x, dict)]

    if isinstance(data, dict):
        for key in ("entries", "items", "contents", "children", "tree"):
            value = data.get(key)
            if isinstance(value, list):
                return [x for x in value if isinstance(x, dict)]

        # Иногда директория может прийти как один объект со вложенными entries
        if data.get("type") in {"dir", "directory"} and isinstance(data.get("entries"), list):
            return [x for x in data["entries"] if isinstance(x, dict)]

    return []


# ---- save_analysis ----
def save_analysis(name: str, content: str) -> str:
    """Сохранить сводку по анализу одного промпта."""
    safe_name = _safe_name(name)
    path = ANALYSES_DIR / f"{safe_name}.md"
    path.write_text(content, encoding="utf-8")
    return f"✅ Анализ сохранён: {path.name} ({len(content)} символов)"


# ---- list_analyses ----
def list_analyses() -> str:
    """Получить список всех сохранённых анализов."""
    files = list(ANALYSES_DIR.glob("*.md"))
    return _format_file_list(
        files,
        empty_message="❌ Нет сохранённых анализов в hw_3_output/analyses",
        title="📋 Сохранённые анализы:",
    )


# ---- read_analysis ----
def read_analysis(name: str) -> str:
    """Прочитать сохранённый анализ промпта по имени."""
    safe_name = _safe_name(name)
    path = ANALYSES_DIR / f"{safe_name}.md"

    if path.exists():
        content = path.read_text(encoding="utf-8")
        return f"═══ АНАЛИЗ: {safe_name} ═══\n\n{content}"

    available = _format_file_list(
        list(ANALYSES_DIR.glob("*.md")),
        empty_message="❌ Файл анализа не найден, и в директории analyses пока нет файлов.",
        title="Доступные анализы:",
    )
    return f"❌ Анализ '{safe_name}' не найден.\n\n{available}"


# ---- save_report ----
def save_report(name: str, content: str) -> str:
    """Сохранить отчёт или сгенерированный промпт."""
    safe_name = _safe_name(name)
    path = REPORTS_DIR / f"{safe_name}.md"
    path.write_text(content, encoding="utf-8")
    return f"✅ Отчёт сохранён: {path.name} ({len(content)} символов)"


# ---- read_report ----
def read_report(name: str) -> str:
    """Прочитать сохранённый отчёт."""
    safe_name = _safe_name(name)
    path = REPORTS_DIR / f"{safe_name}.md"

    if path.exists():
        content = path.read_text(encoding="utf-8")
        return f"═══ ОТЧЁТ: {safe_name} ═══\n\n{content}"

    available = _format_file_list(
        list(REPORTS_DIR.glob("*.md")),
        empty_message="❌ Файл отчёта не найден, и в директории reports пока нет файлов.",
        title="Доступные отчёты:",
    )
    return f"❌ Отчёт '{safe_name}' не найден.\n\n{available}"


# ---- List files in github repo ----
async def list_repo_files(session) -> list[dict]:
    """
    Обойти репозиторий через GitHub MCP и собрать список prompt-файлов.
    Возвращает:
    [{"path": "openai/codex.md", "name": "codex.md", "folder": "openai", "size": ...}]
    """
    REPO = {"owner": "asgeirtj", "repo": "system_prompts_leaks"}

    def _parse_json_text(raw_text: str):
        raw_text = (raw_text or "").strip()
        if not raw_text:
            return None

        fenced = re.match(r"^```(?:json)?\s*(.*?)\s*```$", raw_text, flags=re.DOTALL)
        if fenced:
            raw_text = fenced.group(1).strip()

        try:
            return json.loads(raw_text)
        except Exception:
            return None

    def _extract_entries(data):
        if data is None:
            return []

        if isinstance(data, list):
            return [x for x in data if isinstance(x, dict)]

        if isinstance(data, dict):
            for key in ("entries", "items", "contents", "children", "tree"):
                value = data.get(key)
                if isinstance(value, list):
                    return [x for x in value if isinstance(x, dict)]

            if data.get("type") in {"dir", "directory"} and isinstance(data.get("entries"), list):
                return [x for x in data["entries"] if isinstance(x, dict)]

        return []

    async def _call_tool_text(tool_name: str, args: dict) -> str:
        result = await session.call_tool(tool_name, args)
        parts = []
        for part in getattr(result, "content", []) or []:
            text = getattr(part, "text", None)
            if text:
                parts.append(text)
            else:
                parts.append(str(part))
        return "\n".join(parts).strip()

    async def fetch_dir(dir_path: str):
        """
        Пытаемся получить директорию несколькими совместимыми способами:
        1) path="/"
        2) path=""
        3) вообще без path
        """
        attempts = []

        if dir_path == "":
            attempts = [
                {**REPO, "path": "/"},
                {**REPO, "path": ""},
                {**REPO},
            ]
        else:
            norm = dir_path.strip("/")
            attempts = [
                {**REPO, "path": norm},
                {**REPO, "path": f"/{norm}"},
                {**REPO, "path": dir_path},
            ]

        last_error = None
        for args in attempts:
            try:
                raw_text = await _call_tool_text("get_file_contents", args)
                data = _parse_json_text(raw_text)
                entries = _extract_entries(data)
                if entries:
                    return entries
            except Exception as e:
                last_error = e

        if last_error:
            raise last_error
        return []

    collected: list[dict] = []
    visited: set[str] = set()

    async def walk(dir_path: str = ""):
        if dir_path in visited:
            return
        visited.add(dir_path)

        entries = await fetch_dir(dir_path)

        for item in entries:
            item_type = item.get("type") or item.get("kind") or item.get("mode")
            item_path = item.get("path") or item.get("name") or ""
            item_name = Path(item_path).name if item_path else item.get("name", "")
            item_size = item.get("size")

            if not item_path:
                continue

            is_dir = (
                item_type in {"dir", "directory", "tree"}
                or item_path.endswith("/")
                or item.get("entries") is not None
                or item.get("children") is not None
            )

            if is_dir:
                await walk(item_path.rstrip("/"))
                continue

            if item_name.lower().endswith(".md"):
                lower_path = item_path.lower()

                if any(skip in lower_path for skip in ["/license", "/contributing", "readme.md"]):
                    continue

                folder = Path(item_path).parent.as_posix()
                folder = "" if folder == "." else folder

                collected.append(
                    {
                        "path": item_path,
                        "name": item_name,
                        "folder": folder,
                        "size": item_size,
                    }
                )

    # Сначала пробуем обычный обход директорий
    try:
        await walk("")
    except Exception as e:
        print(f"⚠️ Directory traversal через get_file_contents не сработал: {e}")
        print("↪️ Перехожу на fallback через search_code...")

        # Fallback: ищем markdown-файлы через search_code
        raw_text = await _call_tool_text(
            "search_code",
            {
                "query": "extension:md repo:asgeirtj/system_prompts_leaks",
            },
        )
        data = _parse_json_text(raw_text)

        items = []
        if isinstance(data, dict):
            for key in ("items", "results", "matches"):
                if isinstance(data.get(key), list):
                    items = data[key]
                    break
        elif isinstance(data, list):
            items = data

        seen = set()
        for item in items:
            path = (
                item.get("path")
                or item.get("file_path")
                or item.get("name")
                or ""
            )
            if not path or not path.lower().endswith(".md"):
                continue

            lower_path = path.lower()
            if any(skip in lower_path for skip in ["/license", "/contributing", "readme.md"]):
                continue

            if path in seen:
                continue
            seen.add(path)

            collected.append(
                {
                    "path": path,
                    "name": Path(path).name,
                    "folder": "" if Path(path).parent.as_posix() == "." else Path(path).parent.as_posix(),
                    "size": item.get("size"),
                }
            )

    dedup = {}
    for x in collected:
        dedup[x["path"]] = x

    result = list(dedup.values())
    result.sort(key=lambda x: (x["folder"].lower(), x["name"].lower()))
    return result

print("\n✅ Файловые функции готовы.")


📂 Директория анализов: /content/HSE-Agent-Systems_2026/lectures/lecture_3/hw_3_output/analyses
📂 Директория отчётов:  /content/HSE-Agent-Systems_2026/lectures/lecture_3/hw_3_output/reports

✅ Файловые функции готовы.


In [ ]:
# Опишите каждый инструмент в формате OpenAI function-calling.

FILE_TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "save_analysis",
            "description": (
                "Сохранить анализ одного промпта в директорию hw_3_output/analyses. "
                "Используй после завершения анализа конкретного prompt-файла."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": "Имя файла анализа без расширения .md, например OpenAI_Codex",
                    },
                    "content": {
                        "type": "string",
                        "description": "Полный текст анализа в markdown",
                    },
                },
                "required": ["name", "content"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "list_analyses",
            "description": (
                "Получить список всех сохранённых анализов из директории hw_3_output/analyses. "
                "Используй перед формированием сводного отчёта."
            ),
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_analysis",
            "description": (
                "Прочитать сохранённый анализ по имени из директории hw_3_output/analyses. "
                "Используй при подготовке сводного отчёта."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": "Имя анализа без расширения .md, например OpenAI_Codex",
                    }
                },
                "required": ["name"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "save_report",
            "description": (
                "Сохранить итоговый отчёт или сгенерированный системный промпт "
                "в директорию hw_3_output/reports."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": "Имя файла отчёта без расширения .md, например analysis_report или generated_prompt",
                    },
                    "content": {
                        "type": "string",
                        "description": "Полный текст отчёта или итогового промпта в markdown",
                    },
                },
                "required": ["name", "content"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_report",
            "description": (
                "Прочитать сохранённый отчёт по имени из директории hw_3_output/reports. "
                "Используй перед генерацией нового системного промпта."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": "Имя отчёта без расширения .md, например analysis_report",
                    }
                },
                "required": ["name"],
            },
        },
    },
]


# ============================================================
# Маппинг локальных инструментов
# ============================================================

LOCAL_TOOLS = {
    "read_skill": read_skill,
    "save_analysis": save_analysis,
    "list_analyses": list_analyses,
    "read_analysis": read_analysis,
    "save_report": save_report,
    "read_report": read_report,
}

print(f"\n🔧 Локальные инструменты агента ({len(LOCAL_TOOLS)}):")
for name in LOCAL_TOOLS:
    print(f"  • {name}")

print(f"\n🧩 FILE_TOOL_SCHEMAS: {len(FILE_TOOL_SCHEMAS)}")
for tool in FILE_TOOL_SCHEMAS:
    print(f"  • {tool['function']['name']}")


🔧 Локальные инструменты агента (6):
  • read_skill
  • save_analysis
  • list_analyses
  • read_analysis
  • save_report
  • read_report

🧩 FILE_TOOL_SCHEMAS: 5
  • save_analysis
  • list_analyses
  • read_analysis
  • save_report
  • read_report


---
## Пайплайн — универсальный агент с tool loop

Реализуйте **одного универсального агента** `run_agent()`, который выполняет разные задачи в зависимости от запроса (`Начни анализ`, `сформируй сводку`, `сгенерируй промпт для такой то задачи`).
### Подсказки

- За основу tool loop возьмите `run_reactive_agent` из практики 2 (или аналог из практики 3)
- Используйте `AgentTrace` для отслеживания шагов и токенов

In [ ]:
# ============================================================
# Универсальный агент с tool loop
# ============================================================

# ⚠️ Этап анализа всех промптов может быть долгим, т.к. для каждого файла
# выполняется отдельный агентный вызов с несколькими шагами tool loop.
# Для ускорения вместо синхронных последовательных запросов можно создать
# N потоков (asyncio.gather / ThreadPoolExecutor) и запустить анализ
# нескольких промптов параллельно.
# Как это реализовать — идете на поклон к @Rinaaaa_chan

import os
from copy import deepcopy

AGENT_SYSTEM_PROMPT = f"""Ты универсальный AI-агент для анализа системных промптов.

Ты умеешь работать в трёх сценариях:
1) prompt_analysis — анализ одного или нескольких системных промптов из GitHub-репозитория
2) analysis_report — формирование сводного отчёта по уже сохранённым анализам
3) prompt_generator — генерация нового системного промпта на основе отчёта

Ключевые правила:
- Сначала определи, какой skill подходит под запрос.
- Перед выполнением содержательной работы ОБЯЗАТЕЛЬНО сначала вызови read_skill(name).
- После загрузки skill строго следуй его процедуре шаг за шагом.
- Не выдумывай содержимое файлов и факты: сначала читай данные инструментами, потом делай выводы.
- Для GitHub используй только доступные MCP-инструменты.
- Для промежуточных и итоговых результатов используй локальные файловые инструменты.
- Если задача многоэтапная, выполняй её последовательно.
- Если анализируется несколько prompt-файлов, работай по одному файлу за раз и сохраняй каждый анализ отдельно.
- Когда данных достаточно, дай финальный ответ на русском языке.

═══════════════════════════════════════════
{skills_metadata}
═══════════════════════════════════════════
"""

SKILL_TO_MCP_CANDIDATES = {
    # Для анализа prompt-файлов обычно нужен доступ к содержимому файлов
    "prompt_analysis": [
        "get_file_contents",
        "search_code",
        "search_repositories",
    ],
    # Для отчёта и генерации нового промпта GitHub-инструменты обычно не нужны
    "analysis_report": [],
    "prompt_generator": [],
}

def _tool_result_to_text(result) -> str:
    """Преобразовать результат MCP tool call в текст."""
    parts = []
    for part in getattr(result, "content", []) or []:
        text = getattr(part, "text", None)
        if text is not None:
            parts.append(text)
        else:
            parts.append(str(part))
    return "\n".join(parts).strip() if parts else "{}"


def _try_parse_json(text: str):
    """Аккуратно распарсить JSON, если это JSON; иначе вернуть исходный текст."""
    if not isinstance(text, str):
        return text
    raw = text.strip()
    if not raw:
        return raw

    fenced = re.match(r"^```(?:json)?\s*(.*?)\s*```$", raw, flags=re.DOTALL)
    if fenced:
        raw = fenced.group(1).strip()

    try:
        return json.loads(raw)
    except Exception:
        return text


def _safe_json_loads(raw_args):
    """Безопасно распарсить args из tool_call."""
    if raw_args is None or raw_args == "":
        return {}
    if isinstance(raw_args, dict):
        return raw_args
    if isinstance(raw_args, str):
        return json.loads(raw_args)
    return {}


def _call_local_tool(func_name: str, args: dict):
    """Вызвать локальный инструмент."""
    fn = LOCAL_TOOLS[func_name]
    if args:
        return fn(**args)
    return fn()


def _select_mcp_tools_for_skill(skill_name: str, all_mcp_schemas: dict[str, dict]) -> list[str]:
    """
    Выбрать подмножество MCP-инструментов для skill.
    Если точные кандидаты не найдены, для prompt_analysis делаем мягкий fallback
    на инструменты, связанные с файлами/поиском/репозиторием.
    """
    all_names = set(all_mcp_schemas.keys())
    selected = [name for name in SKILL_TO_MCP_CANDIDATES.get(skill_name, []) if name in all_names]

    if skill_name == "prompt_analysis" and not selected:
        keywords = ("file", "content", "search", "repo", "repository")
        selected = [name for name in all_names if any(k in name.lower() for k in keywords)]

    return sorted(set(selected))


def _should_fan_out_prompt_analysis(question: str) -> bool:
    """
    Нужен ли режим субагентов:
    один отдельный агентный вызов на каждый prompt-файл.
    """
    q = question.lower()

    positive = any(
        marker in q
        for marker in [
            "начни анализ",
            "проанализируй все",
            "проанализируй промпты",
            "анализ всех",
            "анализ репозитория",
            "разбери все",
            "все системные промпты",
        ]
    )

    negative = any(
        marker in q
        for marker in [
            "сводк",
            "отчёт",
            "отчет",
            "сгенерир",
            "новый промпт",
            "generated_prompt",
            "analysis_report",
        ]
    )

    return positive and not negative


def _make_subagent_question(file_info: dict) -> str:
    """
    Сформировать узкий вопрос для анализа одного файла.
    Это и есть 'чистый контекст' на файл.
    """
    path = file_info["path"]
    folder = file_info.get("folder", "")
    name = file_info.get("name", path)

    return (
        "Выполни анализ одного системного промпта из GitHub-репозитория "
        "asgeirtj/system_prompts_leaks.\n\n"
        f"Файл: {path}\n"
        f"Папка: {folder or '[root]'}\n"
        f"Имя файла: {name}\n\n"
        "Действуй так:\n"
        "1. Сначала вызови read_skill('prompt_analysis').\n"
        "2. Затем прочитай именно этот файл через GitHub MCP.\n"
        "3. Выполни структурированный анализ по skill.\n"
        "4. Сохрани результат через save_analysis с осмысленным именем.\n"
        "5. В финале кратко сообщи, какой анализ был сохранён."
    )


def _append_child_trace(parent: AgentTrace, child: AgentTrace, label: str | None = None) -> None:
    """Добавить шаги субагента в родительский trace."""
    if label:
        parent.steps.append(
            AgentStep(
                step_number=len(parent.steps) + 1,
                observations=[label],
            )
        )

    for step in child.steps:
        parent.steps.append(
            AgentStep(
                step_number=len(parent.steps) + 1,
                tool_calls=deepcopy(step.tool_calls),
                observations=deepcopy(step.observations),
                prompt_tokens=step.prompt_tokens,
                cached_tokens=step.cached_tokens,
                context_size=step.context_size,
                context_size_delta=step.context_size_delta,
                completion_tokens=step.completion_tokens,
                total_tokens=step.total_tokens,
                final_answer=step.final_answer,
            )
        )

    parent.prompt_tokens_total += child.prompt_tokens_total
    parent.completion_tokens_total += child.completion_tokens_total
    parent.total_tokens_total += child.total_tokens_total
    parent.latest_context_size = child.latest_context_size
    parent.max_context_size = max(parent.max_context_size, child.max_context_size)
    parent.total_steps = len(parent.steps)


async def _run_agent_loop(
    question: str,
    session,
    all_mcp_schemas: dict[str, dict],
    system_prompt: str,
    max_steps: int,
    verbose: bool,
) -> AgentTrace:
    """
    Один обычный reactive tool loop.
    ВАЖНО: каждый вызов этой функции = новый чистый контекст messages.
    """
    trace = AgentTrace(question=question)

    # На старте агент видит только:
    # - read_skill
    # - локальные файловые инструменты
    # MCP-инструменты подключаются позже после read_skill(...)
    active_tools = [READ_SKILL_TOOL_SCHEMA] + FILE_TOOL_SCHEMAS
    active_tool_names = {"read_skill"} | set(tool["function"]["name"] for tool in FILE_TOOL_SCHEMAS)
    active_mcp_tool_names = set()

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]

    if verbose:
        print(f"[State] Вопрос: {question}")
        print(f"[Tools] На старте доступно: {sorted(active_tool_names)}")

    prev_context_size = 0

    for step_num in range(1, max_steps + 1):
        current_step = AgentStep(step_number=step_num)

        if verbose:
            print(f"\n{'=' * 60}")
            print(f"--- Шаг {step_num} ---")
            print("  [Decision] LLM анализирует состояние...")
            print(f"  [Tools] Доступно: {sorted(active_tool_names)}")

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=active_tools,
            tool_choice="auto",
            temperature=0,
        )

        usage = _extract_usage(response)
        current_step.prompt_tokens = usage["prompt_tokens"]
        current_step.cached_tokens = usage["cached_tokens"]
        current_step.context_size = usage["context_size"]
        current_step.context_size_delta = current_step.context_size - prev_context_size
        current_step.completion_tokens = usage["completion_tokens"]
        current_step.total_tokens = usage["total_tokens"]

        trace.prompt_tokens_total += current_step.prompt_tokens
        trace.completion_tokens_total += current_step.completion_tokens
        trace.total_tokens_total += current_step.total_tokens
        trace.latest_context_size = current_step.context_size
        trace.max_context_size = max(trace.max_context_size, current_step.context_size)

        prev_context_size = current_step.context_size

        if verbose:
            print(
                f"  [Tokens] context={current_step.context_size} "
                f"(Δ {current_step.context_size_delta:+}) "
                f"[cached={current_step.cached_tokens}, miss={current_step.prompt_tokens}], "
                f"completion={current_step.completion_tokens}, total={current_step.total_tokens}"
            )

        msg = response.choices[0].message
        messages.append(msg)

        # --- STOP ---
        if not msg.tool_calls:
            current_step.final_answer = msg.content or ""
            trace.steps.append(current_step)
            trace.total_steps = step_num
            trace.final_answer = msg.content or ""

            if verbose:
                print("  [Decision] → Финальный ответ (нет tool_calls)")
                print(f"  [Result] {trace.final_answer}")

            return trace

        if verbose:
            print(f"  [Decision] → Вызвать {len(msg.tool_calls)} инструмент(ов)")

        # --- ACTION ---
        for tool_call in msg.tool_calls:
            func_name = getattr(tool_call.function, "name", "") or ""
            raw_args = getattr(tool_call.function, "arguments", "{}")

            # Фиксируем вызов
            try:
                args = _safe_json_loads(raw_args)
            except Exception as e:
                args = {"_raw_args": raw_args, "_parse_error": str(e)}

            current_step.tool_calls.append({"name": func_name, "args": args})

            # Защита от битого tool call
            if not func_name.strip():
                result_text = json.dumps(
                    {"error": "Пустое имя инструмента в tool_call"},
                    ensure_ascii=False,
                )
                result_parsed = _try_parse_json(result_text)

                current_step.observations.append(result_parsed)
                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "content": result_text,
                    }
                )

                if verbose:
                    print("  [Action] <empty_tool_call>")
                    print(f"  [Observation] {result_parsed}")

                continue

            # Локальные инструменты
            if func_name in LOCAL_TOOLS:
                try:
                    result = _call_local_tool(func_name, args if isinstance(args, dict) else {})
                    result_text = result if isinstance(result, str) else json.dumps(result, ensure_ascii=False)
                except Exception as e:
                    result_text = json.dumps({"error": str(e)}, ensure_ascii=False)

                if func_name == "read_skill":
                    skill_name = (args or {}).get("name", "")
                    newly_added = []

                    if skill_name in SKILLS:
                        for mcp_name in _select_mcp_tools_for_skill(skill_name, all_mcp_schemas):
                            if mcp_name not in active_tool_names:
                                active_tools.append(all_mcp_schemas[mcp_name])
                                active_tool_names.add(mcp_name)
                                active_mcp_tool_names.add(mcp_name)
                                newly_added.append(mcp_name)

                    if verbose:
                        print(f"  [Action] read_skill({skill_name})")
                        print(f"  [Observation] Level 2 загружен")
                        if newly_added:
                            print(f"  [Tools+] Подключены MCP-инструменты: {newly_added}")
                        print(f"  [Tools] Теперь доступно: {sorted(active_tool_names)}")
                else:
                    if verbose:
                        print(f"  [Action] {func_name}({args})")

            # MCP-инструменты, которые уже раскрыты
            elif func_name in active_mcp_tool_names:
                try:
                    result = await session.call_tool(func_name, args if isinstance(args, dict) else {})
                    result_text = _tool_result_to_text(result)
                except Exception as e:
                    result_text = json.dumps({"error": str(e)}, ensure_ascii=False)

                if verbose:
                    print(f"  [Action] {func_name}({args})")

            # MCP-инструмент существует, но ещё не раскрыт
            elif func_name in all_mcp_schemas:
                result_text = json.dumps(
                    {"error": f"Инструмент '{func_name}' ещё не раскрыт. Сначала загрузи подходящий skill через read_skill()."},
                    ensure_ascii=False,
                )
                if verbose:
                    print(f"  [Action] {func_name}({args})")

            # Вообще неизвестный инструмент
            else:
                result_text = json.dumps({"error": f"Unknown tool: {func_name}"}, ensure_ascii=False)
                if verbose:
                    print(f"  [Action] {func_name}({args})")

            result_parsed = _try_parse_json(result_text)
            current_step.observations.append(result_parsed)

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result_text,
                }
            )

            if verbose:
                print(f"  [Observation] {result_parsed}")

        trace.steps.append(current_step)

        if verbose:
            print(f"  [State_i] Состояние обновлено (messages: {len(messages)})")

    trace.total_steps = max_steps
    trace.final_answer = "[Агент не завершил работу за отведённое число шагов]"
    return trace


async def run_agent(
    question: str,
    system_prompt: str | None = None,
    max_steps: int = 10,
    verbose: bool = True,
    use_subagents: bool = True,
    analysis_file_limit: int | None = None,
) -> AgentTrace:
    """
    Поддерживает:
    - prompt_analysis
    - analysis_report
    - prompt_generator

    Возможности:
    - обычный reactive tool loop
    - маршрутизация вызовов: read_skill / LOCAL_TOOLS / GitHub MCP
    - progressive disclosure:
        * Level 1 skills.md всегда в system prompt
        * Level 2 skill загружается через read_skill
        * MCP tools подключаются только после read_skill
    - режим субагентов для анализа многих prompt-файлов:
        один файл = один отдельный agent run = чистый контекст
    """
    system_prompt = system_prompt or AGENT_SYSTEM_PROMPT

    with open(os.devnull, "w", encoding="utf-8") as errlog:
        async with stdio_client(github_server_params, errlog=errlog) as (read_stream, write_stream):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()

                # DISCOVERY всех MCP-инструментов
                tools_response = await session.list_tools()
                all_mcp_schemas = {
                    t["function"]["name"]: t
                    for t in format_mcp_tools(tools_response.tools)
                }

                if verbose:
                    print(f"🔌 Подключено к GitHub MCP")
                    print(f"🔧 Обнаружено MCP-инструментов: {len(all_mcp_schemas)}")

                # субагенты для массового анализа prompt-файлов
                if use_subagents and _should_fan_out_prompt_analysis(question):
                    parent_trace = AgentTrace(question=question)

                    files = await list_repo_files(session)
                    if analysis_file_limit is not None:
                        files = files[:analysis_file_limit]

                    if verbose:
                        print(f"\n📚 Режим subagent fan-out: найдено файлов для анализа = {len(files)}")
                        if analysis_file_limit is not None:
                            print(f"⚠️ Ограничение analysis_file_limit = {analysis_file_limit}")

                    if not files:
                        parent_trace.final_answer = "Не удалось найти prompt-файлы в репозитории."
                        return parent_trace

                    parent_trace.steps.append(
                        AgentStep(
                            step_number=1,
                            observations=[{
                                "mode": "subagent_fanout",
                                "files_count": len(files),
                                "files": [f["path"] for f in files],
                            }],
                        )
                    )

                    subanswers = []

                    for idx, file_info in enumerate(files, start=1):
                        if verbose:
                            print(f"\n{'#' * 70}")
                            print(f"### Subagent {idx}/{len(files)}: {file_info['path']}")
                            print(f"{'#' * 70}")

                        sub_question = _make_subagent_question(file_info)
                        sub_trace = await _run_agent_loop(
                            question=sub_question,
                            session=session,
                            all_mcp_schemas=all_mcp_schemas,
                            system_prompt=system_prompt,
                            max_steps=max_steps,
                            verbose=verbose,
                        )

                        _append_child_trace(
                            parent_trace,
                            sub_trace,
                            label=f"[SUBAGENT] Анализ файла: {file_info['path']}",
                        )
                        subanswers.append(f"- {file_info['path']}: {sub_trace.final_answer}")

                    parent_trace.total_steps = len(parent_trace.steps)
                    parent_trace.final_answer = (
                        "✅ Завершён пакетный анализ prompt-файлов через субагентов.\n\n"
                        f"Обработано файлов: {len(files)}\n\n"
                        "Краткий итог по subagent run:\n"
                        + "\n".join(subanswers)
                    )
                    return parent_trace

                # одиночный agent run
                return await _run_agent_loop(
                    question=question,
                    session=session,
                    all_mcp_schemas=all_mcp_schemas,
                    system_prompt=system_prompt,
                    max_steps=max_steps,
                    verbose=verbose,
                )

In [ ]:
# ============================================================
# Тест кейс 1. Запрос на начало анализа
# ============================================================

trace_1 = await run_agent(
    question=(
        "Начни анализ системных промптов из GitHub-репозитория. "
        "Для каждого найденного prompt-файла выполни отдельный анализ, "
        "сохрани каждый результат отдельно и в финале сообщи, что было сохранено."
    ),
    max_steps=12,
    verbose=True,
    use_subagents=True,
    analysis_file_limit=3,
)

print("\n" + "=" * 80)
print("SUMMARY TEST 1")
print("=" * 80)
print(trace_1.summary())

print("\n" + "=" * 80)
print("СОХРАНЁННЫЕ АНАЛИЗЫ")
print("=" * 80)
print(list_analyses())


🔌 Подключено к GitHub MCP
🔧 Обнаружено MCP-инструментов: 26

📚 Режим subagent fan-out: найдено файлов для анализа = 3
⚠️ Ограничение analysis_file_limit = 3

######################################################################
### Subagent 1/3: CONTRIBUTING.md
######################################################################
[State] Вопрос: Выполни анализ одного системного промпта из GitHub-репозитория asgeirtj/system_prompts_leaks.

Файл: CONTRIBUTING.md
Папка: [root]
Имя файла: CONTRIBUTING.md

Действуй так:
1. Сначала вызови read_skill('prompt_analysis').
2. Затем прочитай именно этот файл через GitHub MCP.
3. Выполни структурированный анализ по skill.
4. Сохрани результат через save_analysis с осмысленным именем.
5. В финале кратко сообщи, какой анализ был сохранён.
[Tools] На старте доступно: ['list_analyses', 'read_analysis', 'read_report', 'read_skill', 'save_analysis', 'save_report']

--- Шаг 1 ---
  [Decision] LLM анализирует состояние...
  [Tools] Доступно: ['list_anal

In [ ]:
# ============================================================
# Тест кейс 2. Запрос на формирование итоговой сводки
# ============================================================

trace_2 = await run_agent(
    question=(
        "Сформируй сводный отчёт по всем уже сохранённым анализам. "
        "Построй сравнительную таблицу, выдели общие паттерны, сильные и слабые стороны, "
        "сформулируй рекомендации и сохрани результат как analysis_report."
    ),
    max_steps=12,
    verbose=True,
    use_subagents=False,
)

print("\n" + "=" * 80)
print("SUMMARY TEST 2")
print("=" * 80)
print(trace_2.summary())

print("\n" + "=" * 80)
print("СОДЕРЖИМОЕ analysis_report")
print("=" * 80)
report_text = read_report("analysis_report")
print(report_text[:5000])

🔌 Подключено к GitHub MCP
🔧 Обнаружено MCP-инструментов: 26
[State] Вопрос: Сформируй сводный отчёт по всем уже сохранённым анализам. Построй сравнительную таблицу, выдели общие паттерны, сильные и слабые стороны, сформулируй рекомендации и сохрани результат как analysis_report.
[Tools] На старте доступно: ['list_analyses', 'read_analysis', 'read_report', 'read_skill', 'save_analysis', 'save_report']

--- Шаг 1 ---
  [Decision] LLM анализирует состояние...
  [Tools] Доступно: ['list_analyses', 'read_analysis', 'read_report', 'read_skill', 'save_analysis', 'save_report']
  [Tokens] context=1701 (Δ +1701) [cached=0, miss=1701], completion=15, total=1716
  [Decision] → Вызвать 1 инструмент(ов)
  [Action] read_skill(analysis_report)
  [Observation] Level 2 загружен
  [Tools] Теперь доступно: ['list_analyses', 'read_analysis', 'read_report', 'read_skill', 'save_analysis', 'save_report']
  [Observation] ═══ SKILL ЗАГРУЖЕН: Формирование сводного отчёта по анализам промптов ═══

СТРОГО следуй 

In [ ]:
# ============================================================
# Тест кейс 2. Запрос на формирование промпта для случайного ассистента
# ============================================================

trace_3 = await run_agent(
    question=(
        "На основе сохранённого analysis_report сгенерируй новый системный промпт "
        "для AI-ассистента поддержки студентов университета. "
        "Ассистент должен помогать с дедлайнами, курсами, расписанием, навигацией по учебному процессу, "
        "уметь работать с инструментами и честно обрабатывать неопределённость. "
        "Сохрани результат как generated_prompt."
    ),
    max_steps=12,
    verbose=True,
    use_subagents=False,
)

print("\n" + "=" * 80)
print("SUMMARY TEST 3")
print("=" * 80)
print(trace_3.summary())

print("\n" + "=" * 80)
print("СОДЕРЖИМОЕ generated_prompt")
print("=" * 80)
generated_prompt_text = read_report("generated_prompt")
print(generated_prompt_text[:5000])

🔌 Подключено к GitHub MCP
🔧 Обнаружено MCP-инструментов: 26
[State] Вопрос: На основе сохранённого analysis_report сгенерируй новый системный промпт для AI-ассистента поддержки студентов университета. Ассистент должен помогать с дедлайнами, курсами, расписанием, навигацией по учебному процессу, уметь работать с инструментами и честно обрабатывать неопределённость. Сохрани результат как generated_prompt.
[Tools] На старте доступно: ['list_analyses', 'read_analysis', 'read_report', 'read_skill', 'save_analysis', 'save_report']

--- Шаг 1 ---
  [Decision] LLM анализирует состояние...
  [Tools] Доступно: ['list_analyses', 'read_analysis', 'read_report', 'read_skill', 'save_analysis', 'save_report']
  [Tokens] context=1729 (Δ +1729) [cached=0, miss=1729], completion=15, total=1744
  [Decision] → Вызвать 1 инструмент(ов)
  [Action] read_skill(prompt_generator)
  [Observation] Level 2 загружен
  [Tools] Теперь доступно: ['list_analyses', 'read_analysis', 'read_report', 'read_skill', 'save_ana

<img src="pictures\hw_file_1.PNG" width="600"/>

<img src="pictures\hw_file_2.PNG" width="600"/>